In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import sys
sys.path.insert(0, "../func_py/")
import infer_noise as infn
import data_utils as dt

metadata = pd.read_csv('metadata/metadata.tsv', sep='\t', index_col=0)

### Learning the three noise models for each memory sample

The learned parameters are saved in the `inference/noise` folder

In [10]:
# Parameters to set

#  `mem` : samples of memory cells
#  `pb` : samples of plasmablasts
cell='mem'

# Donors:
# `AT`
# `D01`
# `IM`
# `IZ`
# `MRK`
# `MT`
pat='D01'

# Resolution of the log frequency domain for the integration
n_points = 15000

# Number of processes for parallel computing (only negative binomial models)
n_threads = 5

In [ ]:
aux = metadata[metadata.cell == cell]
aux = aux[aux.patient == pat]
samples = aux[aux.n_repls > 1].index
for sample in samples:
    
    print(sample)
    n_uniq, n_counts = dt.import_sample_counts(sample, metadata.loc[sample].n_repls, 'family', 'count')
    
    # Poisson
    infer = infn.infer_noise_poiss(n_uniq, n_counts, False, n_points=n_points)
    infer.run(x0=(2.5, -7), bounds=((2, 3), (-8, -3)))
    infer.compute_errors()
    infer.write_on_file('inference/noise/', sample+'_'+infer.name+'.txt')
    print(infer.result.x, infer.errors[0])
    
    # Neg bin beta = 1
    infer = infn.infer_noise_negbin(n_uniq, n_counts, False, False, n_points=n_points, n_threads=n_threads)
    infer.run(x0=(2.5, -7, 0.2), bounds=((2, 3), (-8, -1), (0.001, 2)))
    infer.compute_errors()
    infer.write_on_file('inference/noise/', sample+'_'+infer.name+'.txt')
    print(infer.result.x, infer.errors[0])
    
    # Neg bin free beta
    infer = infn.infer_noise_negbin(n_uniq, n_counts, True, False, n_points=n_points, n_threads=n_threads)
    infer.ftol = 5 * 10**(-6)
    infer.run(x0=(2.5, -6, 0.1, 1), bounds=((2, 3), (-7, -4), (0.001, 2), (0.2, 3)))
    infer.compute_errors()
    infer.write_on_file('inference/noise/', sample+'_'+infer.name+'.txt')
    print(infer.result.x, infer.errors[0])